# 포트핏 2단계: ETF 전체 구성종목 수집기

지금 계산기는 ETF마다 **상위 10개 종목**만 알고 있어요. 이 노트북은 한국거래소(KRX)가 매일 공개하는 **ETF 납부자산구성내역(PDF)**에서 **전체 구성종목과 비중**을 받아 `holdings.json` 파일로 저장해요. 이 파일을 GitHub에 올리면 계산기가 전체 데이터로 계산하게 돼요.

### 사용법
위에서부터 칸마다 **Shift + Enter**. 칸마다 **"확인할 것"**에 적힌 결과가 나오는지 봐 주세요.
에러가 나면 **에러 메시지 전체를 복사해서** 그대로 알려 주세요.

## 1. 도구 설치

`pykrx`는 한국거래소 데이터를 파이썬으로 불러오는 무료 라이브러리예요. 코랩에는 기본으로 없어서 설치해요. (`!`로 시작하는 줄은 파이썬이 아니라 설치 명령이에요.)

**확인할 것:** 몇 초 뒤 에러 없이 끝나면 성공.

In [ ]:
!pip install -q pykrx
print("설치 완료")

## 1-2. 한국거래소 로그인 정보 넣기

거래소 데이터는 **data.krx.co.kr 일반 회원(아이디·비밀번호 방식)** 로그인이 필요해요. 카카오 등 간편 로그인 계정은 안 돼요.

1. 왼쪽 세로 메뉴의 **열쇠 🔑(보안 비밀)**에서 `KRX_ID`, `KRX_PW` 두 개를 추가하고 **노트북 액세스**를 켜요.
2. 아래 칸을 실행해요. 비밀번호를 코드에 직접 적지 마세요.

**확인할 것:** `로그인 정보 준비 완료`. 로그인은 1시간 뒤 만료되니, 쉬었다 오면 이 칸부터 다시 실행하세요.

In [ ]:
import os
from google.colab import userdata

os.environ["KRX_ID"] = userdata.get("KRX_ID")
os.environ["KRX_PW"] = userdata.get("KRX_PW")
print("로그인 정보 준비 완료")

## 2. 한 개로 먼저 시험하기 ⭐

KODEX 200 하나만 받아 봐요. 여기서 막히면 아래 칸들도 안 되니까, **이 칸 결과를 꼭 알려 주세요.**

**확인할 것:** `종목 수`가 200 안팎이고, 표 맨 위에 삼성전자·SK하이닉스 코드(005930, 000660)가 보이면 성공.

In [ ]:
from pykrx import stock

df = stock.get_etf_portfolio_deposit_file("069500")
print("종목 수:", len(df))
print("열 이름:", list(df.columns))
df.head(10)

## 3. 전체 ETF 받기

포트핏이 지원하는 ETF 20개를 차례로 받아요. 1~3분 걸려요.

- 비중(%) 열이 있으면 그대로 쓰고, 없으면 평가금액으로 비중을 계산해요.
- 국내 종목은 코드(예: 005930)를 이름(삼성전자)으로 바꿔요. 해외 종목은 받은 이름 그대로 둬요.

**확인할 것:** 마지막에 `성공 20개`가 나오면 완벽. 실패한 ETF가 있으면 그 줄을 알려 주세요.

In [ ]:
import time

ETFS = {
    "069500": "KODEX 200",            "102110": "TIGER 200",           "229200": "KODEX 코스닥150",
    "396500": "TIGER 반도체TOP10",     "091160": "KODEX 반도체",         "305720": "KODEX 2차전지산업",
    "364980": "TIGER 2차전지TOP10",    "360750": "TIGER 미국S&P500",     "379800": "KODEX 미국S&P500",
    "360200": "ACE 미국S&P500",        "379780": "RISE 미국S&P500",      "379810": "KODEX 미국나스닥100",
    "367380": "ACE 미국나스닥100",      "381170": "TIGER 미국테크TOP10",   "465580": "ACE 미국빅테크TOP7 Plus",
    "381180": "TIGER 미국필라델피아반도체나스닥", "390390": "KODEX 미국반도체",
    "458730": "TIGER 미국배당다우존스",  "446720": "SOL 미국배당다우존스",  "489250": "KODEX 미국배당다우존스",
}

def get_holdings(code):
    df = stock.get_etf_portfolio_deposit_file(code)
    if df is None or len(df) == 0:
        raise ValueError("빈 결과가 왔어요")
    w = df["비중"].astype(float)
    if w.sum() < 50:                      # 비중이 비어 있으면 금액으로 계산
        a = df["금액"].astype(float)
        w = a / a.sum() * 100 if a.sum() > 0 else w * 0
    rows = []
    for ticker, weight in w.items():
        if weight > 0:
            name = str(df.loc[ticker, "구성종목명"]).strip() or str(ticker)
            rows.append([str(ticker).strip(), name, round(float(weight), 4)])
    rows.sort(key=lambda r: -r[2])
    return rows

result, failed = {}, {}
for code, name in ETFS.items():
    try:
        rows = get_holdings(code)
        result[code] = {"name": name, "holdings": rows}
        top = ", ".join(f"{r[1]} {r[2]:.1f}" for r in rows[:3])
        print(f"OK  {name:<24} 종목 {len(rows):>4}개 · 비중 합 {sum(r[2] for r in rows):6.1f}% · {top}")
    except Exception as e:
        failed[code] = str(e)
        print(f"실패 {name:<24} {e}")
    time.sleep(0.5)

print(f"\n성공 {len(result)}개 / 실패 {len(failed)}개")
print("참고: 미국 ETF는 거래소가 수량만 주고 비중을 0으로 줘서 종목 0개로 나와요. 계산기에서는 상위 10개 데이터를 써요.")

## 4. 해외 종목 이름 확인

미국 ETF 안의 종목 이름이 어떤 모양으로 왔는지 봐요 (영문 이름인지, 코드인지). 계산기에서 '엔비디아' 같은 한글 이름과 맞추려면 이 모양을 알아야 해요.

**확인할 것:** 출력된 10줄을 그대로 복사해서 알려 주세요.

In [ ]:
sample = result.get("360750") or next(iter(result.values()))
for t, n, w in sample["holdings"][:10]:
    print(t, "|", n, "|", w)

## 4-2. 미국 ETF 빼고 업종 붙이기

비중이 없는 미국 ETF를 빼고, 국내 종목마다 거래소 업종(전기·전자, 금융 등)을 붙여요.

**확인할 것:** `저장할 ETF: 7 개`, `업종 붙이기 완료`. 업종 날짜(`20261023`)는 최근 영업일로 바꿔 쓰세요.

In [ ]:
result = {k: v for k, v in result.items() if v["holdings"]}
print("저장할 ETF:", len(result), "개")

sectors = {}
for mkt in ["KOSPI", "KOSDAQ"]:
    try:
        s = stock.get_market_sector_classifications("20261023", mkt)
        sectors.update(s["업종명"].astype(str).to_dict())
        print(mkt, "업종 분류", len(s), "개")
    except Exception as e:
        print(mkt, "업종 분류 실패:", e)

for v in result.values():
    for r in v["holdings"]:
        if len(r) == 3:
            r.append(sectors.get(r[0], ""))
print("업종 붙이기 완료")

## 5. 파일로 저장하고 내려받기

`holdings.json` 파일이 만들어지고 자동으로 내려받기가 시작돼요. 이 파일을 다음 단계에서 GitHub에 올려요.

In [ ]:
import json, datetime

data = {
    "asof": datetime.date.today().isoformat(),
    "source": "한국거래소 ETF 납부자산구성내역 (pykrx)",
    "etfs": result,
}
with open("holdings.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False)

print("저장 완료:", len(result), "개 ETF,", sum(len(v["holdings"]) for v in result.values()), "줄")
try:
    from google.colab import files
    files.download("holdings.json")
except ImportError:
    print("코랩이 아니면 왼쪽 파일 목록에서 holdings.json을 직접 내려받으세요.")

## (막혔을 때만) 거래소 직접 접속 진단

2번 칸에서 에러가 났거나 `종목 수: 0`이 나왔을 때만 실행하세요. 거래소 서버가 어떤 답을 주는지 보여줘요. **출력 전체를 복사해서 알려 주세요.**

In [ ]:
import requests, datetime

r = requests.post(
    "http://data.krx.co.kr/comm/bldAttendant/getJsonData.cmd",
    data={"bld": "dbms/MDC/STAT/standard/MDCSTAT05001", "locale": "ko_KR",
          "trdDd": datetime.date.today().strftime("%Y%m%d"), "isuCd": "KR7069500007",
          "share": "1", "money": "1", "csvxls_isNo": "false"},
    headers={"User-Agent": "Mozilla/5.0",
             "Referer": "http://data.krx.co.kr/contents/MDC/MDI/mdiLoader/index.cmd?menuId=MDC0201"},
    timeout=20,
)
print("상태 코드:", r.status_code)
print(r.text[:600])